In [1]:
import pandas as pd
from dnsdumpster.DNSDumpsterAPI import DNSDumpsterAPI
import sublist3r
import os

[!] Error: Coloring libraries not installed, no coloring will be used [Check the readme]


In [2]:
# configurações do alvo
target_domain = 'mj.gov.br' 

print(f"--- Iniciando Reconhecimento para: {target_domain} ---")


--- Iniciando Reconhecimento para: mj.gov.br ---


In [3]:
print("[+] Consultando DNSDumpster...")
try:
    results_dns = DNSDumpsterAPI().search(target_domain)
    # extrai apenas os nomes de domínio (host) dos registros encontrados
    dnsdumpster_hosts = [entry['domain'] for entry in results_dns['dns_records']['host']]
except Exception as e:
    print(f"Erro no DNSDumpster: {e}")
    dnsdumpster_hosts = []
print(f"\n[!] Total encontrado no DNSDumpster: {len(dnsdumpster_hosts)}")

[+] Consultando DNSDumpster...

[!] Total encontrado no DNSDumpster: 4


In [4]:
def get_sublist3r_hosts(domain):
    print(f"[+] Consultando Sublist3r para {domain}...")
    try:
        # utilizamos os motores nativos do sublist3r (exceto dnsdumpster, pois já o consultamos,
        # e ele internamente está causando um erro de index devido à atualização do site)
        motores = 'baidu,yahoo,google,bing,ask,netcraft,virustotal,threatcrowd,ssl,passivedns'
        subdomains = sublist3r.main(domain, 40, savefile=None, ports=None, silent=True, verbose=False, enable_bruteforce=False, engines=motores)
        return list(set(subdomains)) if subdomains else []
    except Exception as e:
        print(f"[-] Erro no Sublist3r: {e}")
        return []


In [5]:
sublist3r_hosts = get_sublist3r_hosts(target_domain)
print(f"Total encontrado no Sublist3r: {len(sublist3r_hosts)}")

[+] Consultando Sublist3r para mj.gov.br...
Total encontrado no Sublist3r: 12


In [6]:
# carrega dados historicos


hist_file = r"C:\Users\john.araujo\OneDrive - unb.br\PC SIA\arquivos_projeto_pratico\Colab Notebooks-20260121T122702Z-1-001\Colab Notebooks\unb\unb projeto\sublist3r.txt"
hist_subdomains = []
if os.path.exists(hist_file):
    with open(hist_file, 'r', encoding='utf-8') as f:
        hist_subdomains = [line.strip() for line in f if line.strip()]
    print(f"[+] Total de {len(hist_subdomains)} subdomínios da coleta histórica oficial (Kali Linux).")
else:
    print("[-] Arquivo histórico não encontrado no caminho fornecido.")


[+] Total de 509 subdomínios da coleta histórica oficial (Kali Linux).


In [7]:
# une os resultados das execuções em tempo real com o dataset histórico oficial e garante a lista final idêntica/completa.
all_subdomains = sorted(list(set(dnsdumpster_hosts + sublist3r_hosts + hist_subdomains)))
print(f"\n[!] Total de ativos únicos combinados: {len(all_subdomains)}")


[!] Total de ativos únicos combinados: 520


In [8]:
# cria um dataframe para melhor visualização e exportação
df_assets = pd.DataFrame(all_subdomains, columns=['URL_Ativo'])
df_assets['Organizacao'] = target_domain
print(df_assets.head(10))

                  URL_Ativo Organizacao
0        aaedepen.mj.gov.br   mj.gov.br
1         academi.mj.gov.br   mj.gov.br
2         academy.mj.gov.br   mj.gov.br
3       acessovpn.mj.gov.br   mj.gov.br
4  admin.monitora.mj.gov.br   mj.gov.br
5      agente.dev.mj.gov.br   mj.gov.br
6    agente.stage.mj.gov.br   mj.gov.br
7     alterasenha.mj.gov.br   mj.gov.br
8      anpd-super.mj.gov.br   mj.gov.br
9             apm.mj.gov.br   mj.gov.br


In [9]:
# salva a lista de alvos para a fase de scan de vulnerabilidades
df_assets.to_csv('lista_alvos_recon.csv', index=False)
print("\n[OK] Lista de alvos salva em 'lista_alvos_recon.csv'.")


[OK] Lista de alvos salva em 'lista_alvos_recon.csv'.
